<div style="text-align: center;"> 
    <span style="font-size: 40px; font-weight: bold">
        Lập trình cho Khoa học dữ liệu - CQ2023/21
    </span>
</div>

<div style="text-align: center;"> 
    <span style="font-size: 24px; font-weight: bold">
        <i>
            03. Modeling
        </i>
    </span>
</div>

<div style="text-align: center;"> 
    <span style="font-size: 18px">
            Lê Võ Nhật Minh - 22120210
    </span>
</div>

<div style="text-align: right;">
    <span style="font-size: 14px">
        <i>
            Nov 29, 2025
        </i>
    </span>
</div>

---

### **I. Tải lại dữ liệu đã được tiền xử lý**

In [1]:
import numpy as np

data = np.load('../data/processed/dataset.npz', allow_pickle=True)

# Lấy lại từng trường
user_idx = data['user_idx']
user_map = data['user_map']
prod_idx = data['prod_idx']
prod_map = data['prod_map']
rating_col = data['rating_col']
time_col = data['time_col']

print("Load xong!")


Load xong!


### **II. Chia train/test**

Chia train test cho tập dữ liệu. Ở đây ta sẽ lấy rating gần nhất của người dùng để làm test set. Phần dữ liệu còn lại của người dùng sẽ được sử dụng làm train set.

In [ ]:
import numpy as np

# np.lexsort sắp xếp theo key cuối cùng trước (Primary Key là key cuối trong tuple).
# Ta muốn: Sắp theo user_idx (Primary) -> Sau đó đến time_col (Secondary).
# Kết quả: Các user giống nhau nằm cạnh nhau, thời gian tăng dần.
sort_order = np.lexsort((time_col, user_idx))

# Lấy ra mảng user đã được sắp xếp theo trật tự trên
sorted_users = user_idx[sort_order]

# Nếu user[i] != user[i+1], nghĩa là user[i] là giao dịch cuối cùng của user đó.
# Phần tử cuối cùng của mảng luôn là giao dịch cuối của user cuối cùng.

# Tạo mask False toàn bộ
is_latest_mask = np.empty(len(user_idx), dtype=bool)

# So sánh mảng user (từ đầu đến kế cuối) với mảng user (từ thứ 2 đến cuối)
# Nếu khác nhau -> True (Đó là điểm gãy, tức là điểm cuối của user hiện tại)
is_latest_mask[:-1] = sorted_users[:-1] != sorted_users[1:]

# Phần tử cuối cùng luôn là Latest của user cuối cùng
is_latest_mask[-1] = True

# Lấy index thực sự từ mảng sort_order dựa trên mask
test_indices = sort_order[is_latest_mask]   # Chỉ lấy những dòng là latest
train_indices = sort_order[~is_latest_mask] # Lấy phần còn lại (Lịch sử cũ)

# Bây giờ dùng index để cắt mảng gốc
user_train = user_idx[train_indices]
prod_train = prod_idx[train_indices]
rating_train = rating_col[train_indices]

user_test = user_idx[test_indices]
prod_test = prod_idx[test_indices]
rating_test = rating_col[test_indices]

print(f"Tổng dữ liệu: {len(user_idx)}")
print(f"Train set:    {len(user_train)}")
print(f"Test set:     {len(user_test)} (Tương đương số lượng User unique)")

Tổng dữ liệu: 2023070
Train set:    1773796
Test set:     249274 (Tương đương số lượng User unique)
Test set (Users đã có lịch sử trong train): 145790


### **III. Huấn luyện mô hình**

Thực hiện train mô hình và đánh giá kết quả

In [5]:
import scipy.sparse as sparse
from implicit.evaluation import precision_at_k, mean_average_precision_at_k
from implicit.als import AlternatingLeastSquares

# 1. Xác định kích thước ma trận toàn cục
# (Bắt buộc phải dùng kích thước của toàn bộ dữ liệu, không được lấy max của tập train/test)
# Vì tập test có thể chứa user/item chưa từng xuất hiện trong tập train (nếu chia random)
# Nhưng với cách chia Latest Transaction, user tập test chắc chắn nằm trong tập user toàn cục.
num_users = len(user_map)  # Hoặc user_idx.max() + 1
num_items = len(prod_map)  # Hoặc prod_idx.max() + 1


# 2. Chuyển đổi Train/Test từ NumPy Array sang CSR Matrix
# Cấu trúc: (Users x Items)
train_csr = sparse.csr_matrix(
    (rating_train, (user_train, prod_train)), 
    shape=(num_users, num_items)
)

test_csr = sparse.csr_matrix(
    (rating_test, (user_test, prod_test)), 
    shape=(num_users, num_items)
)

print(f"Train matrix shape: {train_csr.shape}")
print(f"Test matrix shape:  {test_csr.shape}")

# 3. Huấn luyện Model (ALS)
# Lưu ý: implicit 0.6.0+ hỗ trợ fit trực tiếp trên User-Item matrix (nếu set use_native=True hoặc mặc định)
# Tuy nhiên, một số phiên bản cũ hoặc setting tối ưu có thể cần Item-User. 
# Để an toàn và chuẩn nhất với tài liệu hiện tại:
model = AlternatingLeastSquares(
    factors=64, 
    regularization=0.1, 
    iterations=15,
    random_state=42
)

# Fit trên tập Train
model.fit(train_csr)

# 4. Đánh giá bằng hàm có sẵn
# Tại sao hàm này cần cả 'train_csr' và 'test_csr'?
# -> Để nó biết user đã xem cái gì ở tập Train rồi mà BỎ QUA không gợi ý lại những cái đó nữa.
# -> Nó chỉ tính điểm nếu model gợi ý trúng những item nằm trong 'test_csr'.

print("Đang đánh giá Precision@10...")
p_at_10 = precision_at_k(
    model, 
    train_user_items=train_csr, 
    test_user_items=test_csr, 
    K=10,
    show_progress=True
)

print("Đang đánh giá MAP@10...")
map_at_10 = mean_average_precision_at_k(
    model, 
    train_user_items=train_csr, 
    test_user_items=test_csr, 
    K=10,
    show_progress=True
)

print("-" * 30)
print(f"Precision@10: {p_at_10:.4f}")
print(f"MAP@10:       {map_at_10:.4f}")

Train matrix shape: (1210271, 249274)
Test matrix shape:  (1210271, 249274)


100%|██████████| 15/15 [05:55<00:00, 23.72s/it]


Đang đánh giá Precision@10...


100%|██████████| 249274/249274 [07:50<00:00, 529.99it/s]


Đang đánh giá MAP@10...


100%|██████████| 249274/249274 [08:43<00:00, 475.89it/s]

------------------------------
Precision@10: 0.0000
MAP@10:       0.0000


***Insight:***
- Hiệu suất của mô hình là 0 do ma trận quá thưa khiến mô hình không thể dự đoán được.
- Có thể tồn tại một số lượng lớn người dùng chỉ mua sản phẩm một lần. Do đó không có dữ liệu lịch sử để có thể tiến hành dự đoán

In [6]:
# Chạy thử đoạn này xem có ra số > 0 không
from implicit.evaluation import train_test_split
train_chk, test_chk = train_test_split(train_csr + test_csr, train_percentage=0.8)
model.fit(train_chk)
print(precision_at_k(model, train_chk, test_chk, K=10))

100%|██████████| 47838/47838 [01:52<00:00, 426.40it/s]

4.180776788327271e-05
